In [0]:
# Databricks notebook source
# 05_web_events_autoloader

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# =========================================================
# CONFIG
# =========================================================
LANDING_PATH = "/Volumes/workspace/00_landing/landing/web_events"
CHECKPOINT_PATH = "/Volumes/workspace/00_landing/landing/checkpoints/web_events"

# sửa lại nếu bronze table của bạn nằm chỗ khác
BRONZE_WEB_TABLE = "workspace.01_bronze.bronze_web_events"
# ví dụ nếu bạn dùng schema layer:
# BRONZE_WEB_TABLE = "workspace.`01_bronze`.web_events"

schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_time", StringType()),
    StructField("session_id", StringType()),
    StructField("user_id", StringType()),
    StructField("event_type", StringType()),
    StructField("product_id", StringType()),
    StructField("order_id", StringType()),
    StructField("quantity", IntegerType()),
    StructField("price_at_event", DoubleType()),
    StructField("page_url", StringType()),
    StructField("referrer", StringType()),
    StructField("utm_source", StringType()),
    StructField("utm_medium", StringType()),
    StructField("utm_campaign", StringType()),
    StructField("campaign_id", StringType()),
    StructField("device", StringType()),
    StructField("browser", StringType()),
    StructField("store_channel", StringType())
])

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load(LANDING_PATH)
)

stream_clean = (
    stream_df
    .withColumn("event_time", F.to_timestamp("event_time", "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .withColumn("ingestion_time", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

def upsert_web_events(microBatchDF, batchId):
    microBatchDF = (
        microBatchDF
        .filter(F.col("event_id").isNotNull())
        .dropDuplicates(["event_id"])
    )

    delta_table = DeltaTable.forName(spark, BRONZE_WEB_TABLE)

    (
        delta_table.alias("t")
        .merge(
            microBatchDF.alias("s"),
            "t.event_id = s.event_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    stream_clean.writeStream
    .foreachBatch(upsert_web_events)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("web_events autoloader completed")